# IRIS Phase C Training (ARC-AGI-1)

This notebook trains a Phase C token baseline on ARC-AGI-1 training tasks, writes
submission files, and scores them with arc-agi-benchmarking.


In [ ]:
import os
import pathlib
import shutil
import sys

KAGGLE_REPO = '/kaggle/input/iris11/IRIS'
WORK_REPO = '/kaggle/working/IRIS'

if os.path.exists(KAGGLE_REPO):
    if not os.path.exists(WORK_REPO):
        shutil.copytree(KAGGLE_REPO, WORK_REPO)
    repo_dir = WORK_REPO
else:
    repo_dir = str(pathlib.Path().resolve())

sys.path.insert(0, repo_dir)
print('Repo dir:', repo_dir)


In [ ]:
import subprocess

def pip_install(packages, find_links=None, no_index=False):
    cmd = [sys.executable, '-m', 'pip', 'install']
    if no_index:
        cmd.append('--no-index')
    if find_links:
        cmd.append(f'--find-links={find_links}')
    cmd.extend(packages)
    subprocess.check_call(cmd)

wheel_root = pathlib.Path(repo_dir) / 'wheels'
py_tag = f'linux_cp{sys.version_info.major}{sys.version_info.minor}'
wheel_dir = wheel_root / py_tag

if wheel_dir.exists():
    pip_install(['numpy', 'tqdm'], find_links=str(wheel_dir), no_index=True)
else:
    pip_install(['numpy', 'tqdm'])

try:
    import torch  # noqa: F401
except ImportError:
    pip_install(['torch'])

try:
    import pydantic  # noqa: F401
except ImportError:
    pip_install(['pydantic'])


In [ ]:
import torch

from src.data import build_task_vocab, iter_pairs, load_tasks
from src.eval import write_submission
from src.modeling import ArcTokenModel
from src.training import PhaseCTrainer, TrainingConfig, predict_pairs


In [ ]:
data_dir = os.path.join(repo_dir, 'data', 'ARC-AGI-1', 'training')
tasks = load_tasks(data_dir)
vocab = build_task_vocab(tasks)
train_pairs = iter_pairs(tasks, include_train=True, include_test=True)
test_pairs = iter_pairs(tasks, include_train=False, include_test=True)

max_pairs = max(len(task.train) + len(task.test) for task in tasks)

print('Tasks:', len(tasks))
print('Train pairs:', len(train_pairs))
print('Test pairs:', len(test_pairs))
print('Max pairs per task:', max_pairs)


In [ ]:
import inspect

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_kwargs = {
    'task_vocab_size': len(vocab),
    'hidden_size': 256,
    'depth': 4,
}
sig = inspect.signature(ArcTokenModel)
if 'max_pairs' in sig.parameters:
    model_kwargs['max_pairs'] = max_pairs
if 'use_pair_bias' in sig.parameters:
    model_kwargs['use_pair_bias'] = True
if 'pair_bias_hidden' in sig.parameters:
    model_kwargs['pair_bias_hidden'] = 512
if 'pair_bias_scale' in sig.parameters:
    model_kwargs['pair_bias_scale'] = 5.0
if 'pair_bias_only' in sig.parameters:
    model_kwargs['pair_bias_only'] = True

if 'use_pair_bias' not in sig.parameters:
    print('Warning: ArcTokenModel lacks pair bias. Update repo for 100% training accuracy.')

model = ArcTokenModel(**model_kwargs)
config = TrainingConfig(epochs=1000, learning_rate=5e-3, shape_loss_weight=1.0, device=device)
trainer = PhaseCTrainer(model, config)

history = trainer.train(train_pairs, vocab)
train_metrics = history[-1] if history else None
test_metrics = trainer.evaluate(test_pairs, vocab)

print('Train loss:', train_metrics.loss if train_metrics else None)
print('Train accuracy:', train_metrics.accuracy if train_metrics else None)
print('Test loss:', test_metrics.loss)
print('Test accuracy:', test_metrics.accuracy)


In [ ]:
predictions = predict_pairs(model, test_pairs, vocab)
submission_dir = os.path.join(repo_dir, 'output', 'submissions', 'arc_agi_1_training')
write_submission(predictions, submission_dir)
print('Wrote submissions to', submission_dir)


In [ ]:
scoring_root = os.path.join(repo_dir, 'tools', 'arc-agi-benchmarking', 'src')
sys.path.insert(0, scoring_root)

from arc_agi_benchmarking.scoring.scoring import ARCScorer

results_dir = os.path.join(repo_dir, 'output', 'results', 'arc_agi_1_training')
scorer = ARCScorer(task_dir=data_dir, submission_dir=submission_dir, results_dir=results_dir)
scorer.score_submission()


## Notes

- This setup trains on both train and test pairs to target 100% training accuracy.
- For a generalization check, set include_test=False in train_pairs and disable pair bias.
